# Credit Risk Model — Notebook 2: Preprocessing

**Goal:** Transform the cleaned EDA output into model-ready arrays.

**Steps:**
- Clean categorical columns (term, emp_length) into usable form
- Impute remaining missing values
- Encode categorical features
- Stratified train/test split
- Apply SMOTE to the training set only (avoiding leakage)

**Input:** data/processed/lending_club_eda.csv  
**Output:** train/test arrays saved for the modeling notebook

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")


def find_repo_root(marker=".git"):
    current = Path.cwd()
    for directory in [current] + list(current.parents):
        if (directory / marker).exists():
            return directory
    raise FileNotFoundError(f"No '{marker}' found above {current}")


REPO_ROOT = find_repo_root()
PROJECT   = REPO_ROOT / "Credit_Risk_Model"
DATA_PROC = PROJECT / "data" / "processed"
sys.path.insert(0, str(PROJECT))

df = pd.read_csv(DATA_PROC / "lending_club_eda.csv")
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded 1,373,915 rows x 26 columns


In [11]:
df.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,purpose,annual_inc,dti,delinq_2yrs,...,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,emp_length,home_ownership,verification_status,target,fico_avg
0,3600.0,36 months,13.99,123.03,C,C4,debt_consolidation,55000.0,5.91,0.0,...,2765.0,29.7,13.0,1.0,0.0,10+ years,MORTGAGE,Not Verified,0,677.0
1,24700.0,36 months,11.99,820.28,C,C1,small_business,65000.0,16.06,1.0,...,21470.0,19.2,38.0,4.0,0.0,10+ years,MORTGAGE,Not Verified,0,717.0
2,20000.0,60 months,10.78,432.66,B,B4,home_improvement,63000.0,10.78,0.0,...,7869.0,56.2,18.0,5.0,0.0,10+ years,MORTGAGE,Not Verified,0,697.0
3,10400.0,60 months,22.45,289.91,F,F1,major_purchase,104433.0,25.37,1.0,...,21929.0,64.5,35.0,6.0,0.0,3 years,MORTGAGE,Source Verified,0,697.0
4,11950.0,36 months,13.44,405.18,C,C3,debt_consolidation,34000.0,10.20,0.0,...,8822.0,68.4,6.0,0.0,0.0,4 years,RENT,Source Verified,0,692.0


In [12]:
# Extract digits from term regardless of spacing: "36 months" -> 36
df["term"] = df["term"].str.extract(r"(\d+)").astype(int)

emp_map = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3,
    "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7,
    "8 years": 8, "9 years": 9, "10+ years": 10,
}
df["emp_length"] = df["emp_length"].map(emp_map)

# Convert credit-line date to years of credit history
df["earliest_cr_line"] = pd.to_datetime(
    df["earliest_cr_line"], format="%b-%Y", errors="coerce"
)
df["credit_history_yrs"] = 2018 - df["earliest_cr_line"].dt.year
df = df.drop(columns=["earliest_cr_line"])

df.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,purpose,annual_inc,dti,delinq_2yrs,...,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,emp_length,home_ownership,verification_status,target,fico_avg,credit_history_yrs
0,3600.0,36,13.99,123.03,C,C4,debt_consolidation,55000.0,5.91,0.0,...,29.7,13.0,1.0,0.0,10.0,MORTGAGE,Not Verified,0,677.0,15.0
1,24700.0,36,11.99,820.28,C,C1,small_business,65000.0,16.06,1.0,...,19.2,38.0,4.0,0.0,10.0,MORTGAGE,Not Verified,0,717.0,19.0
2,20000.0,60,10.78,432.66,B,B4,home_improvement,63000.0,10.78,0.0,...,56.2,18.0,5.0,0.0,10.0,MORTGAGE,Not Verified,0,697.0,18.0
3,10400.0,60,22.45,289.91,F,F1,major_purchase,104433.0,25.37,1.0,...,64.5,35.0,6.0,0.0,3.0,MORTGAGE,Source Verified,0,697.0,20.0
4,11950.0,36,13.44,405.18,C,C3,debt_consolidation,34000.0,10.20,0.0,...,68.4,6.0,0.0,0.0,4.0,RENT,Source Verified,0,692.0,31.0


In [13]:
X = df.drop(columns=["target", "fico_range_low", "fico_range_high"])
y = df["target"]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

print(f"Numeric features    : {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"\nCategorical columns: {cat_cols}")

Numeric features    : 18
Categorical features: 5

Categorical columns: ['grade', 'sub_grade', 'purpose', 'home_ownership', 'verification_status']


#### Preprocessing Pipeline

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ],
    sparse_threshold=0
)

#### Stratified train/test split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y,
)

print(f"Train: {len(X_train):,} rows ({y_train.mean()*100:.1f}% default)")
print(f"Test : {len(X_test):,} rows ({y_test.mean()*100:.1f}% default)")

Train: 1,099,132 rows (21.5% default)
Test : 274,783 rows (21.5% default)


#### Fit Preprocessing

In [16]:
# Sample data for faster experimentation
"""
 from sklearn.model_selection import train_test_split as _tts
X_train, _, y_train, _ = _tts(
    X_train, y_train, train_size=400_000,
    stratify=y_train, random_state=42
)
"""

'\n from sklearn.model_selection import train_test_split as _tts\nX_train, _, y_train, _ = _tts(\n    X_train, y_train, train_size=400_000,\n    stratify=y_train, random_state=42\n)\n'

In [17]:
from imblearn.over_sampling import SMOTE

# Fit on train, transform both
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test) 

# Safety check — output must be dense numeric, not sparse/object
assert X_train_proc.dtype != object, "Preprocessor output is object dtype!"
print(f"Processed feature count: {X_train_proc.shape[1]}")
print(f"Output dtype: {X_train_proc.dtype}")

# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_proc, y_train)

print(f"\nBefore SMOTE: {np.bincount(y_train)}")
print(f"After SMOTE : {np.bincount(y_train_bal)}")

Processed feature count: 83
Output dtype: float64

Before SMOTE: [862991 236141]
After SMOTE : [862991 862991]


In [18]:
import joblib

# Save the fitted preprocessor
joblib.dump(preprocessor, DATA_PROC / "preprocessor.joblib")

# Save the arrays.
np.savez_compressed(
    DATA_PROC / "model_arrays.npz",
    X_train_orig = X_train_proc,          # imbalanced, for class weights
    y_train_orig = np.asarray(y_train),
    X_train_bal  = X_train_bal,           # SMOTE-balanced
    y_train_bal  = y_train_bal,
    X_test_proc  = X_test_proc,
    y_test       = np.asarray(y_test),
)

feature_names = preprocessor.get_feature_names_out()
np.save(DATA_PROC / "feature_names.npy", feature_names)

print(f"Saved both training sets + {len(feature_names)} feature names")
print(f"Imbalanced train: {X_train_proc.shape} | SMOTE train: {X_train_bal.shape}")

Saved both training sets + 83 feature names
Imbalanced train: (1099132, 83) | SMOTE train: (1725982, 83)
